# 🎵 Music Genre Classification using Deep Learning (CNN & Librosa)

### **Objective**
Predict the musical genre of an audio file using Mel-Frequency Cepstral Coefficients (MFCC) and a 2D Convolutional Neural Network (CNN).

### **Workflow Steps**
1. **Audio Fundamentals:** Load audio, visualize Waveform, Mel-Spectrogram, and MFCCs.
2. **Data Preprocessing:** Slice 30s tracks into 3s windows and extract 40 MFCC features.
3. **Train/Val Split:** 80/20 stratified split and reshape into 4D image-like tensors `(batch, 130, 40, 1)`.
4. **Model Architecture:** Build a 2D CNN in Keras using the Functional API (compatible with Keras 2 and Keras 3).
5. **Training:** Train with Early Stopping, ReduceLROnPlateau, and Batch Normalization.
6. **Evaluation:** Plot Loss/Accuracy curves, Confusion Matrix, and Classification Report.
7. **Inference Demo:** Upload/test any audio track, predict genre with confidence score, and render visual reports.

## 1. Import Libraries & Verify Environment

In [ ]:
import os
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf

import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

print(f"TensorFlow Version : {tf.__version__}")
print(f"Librosa Version    : {librosa.__version__}")
print(f"NumPy Version      : {np.__version__}")

## 2. Audio Exploration: Waveform, Mel-Spectrogram & MFCC

We convert 1D time-domain audio waves into 2D acoustic representations:
- **Waveform:** Amplitude vs Time.
- **Mel-Spectrogram:** Frequency energy mapped to human hearing perception (dB scale).
- **MFCC (Mel-Frequency Cepstral Coefficients):** Captures the timbral texture of instruments and vocals.

In [ ]:
def inspect_audio_sample(audio_path=None):
    """Plots Waveform, Mel-Spectrogram, and MFCC for an audio file."""
    # If no audio provided or file missing, generate a sample harmonic audio signal
    if audio_path is None or not os.path.exists(audio_path):
        print("Generating sample audio signal for visualization...")
        sr = 22050
        duration = 5.0
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        signal = 0.5 * np.sin(2 * np.pi * 220 * t) + 0.3 * np.sin(2 * np.pi * 440 * t) + 0.05 * np.random.normal(0, 1, len(t))
        sample_name = "Synthetic Sample (A3 + A4)"
    else:
        signal, sr = librosa.load(audio_path, sr=22050)
        sample_name = os.path.basename(audio_path)
        
    fig, axs = plt.subplots(3, 1, figsize=(12, 10))
    
    # 1. Waveform
    librosa.display.waveshow(signal, sr=sr, ax=axs[0], color='#2563eb')
    axs[0].set_title(f"1. Waveform ({sample_name})", fontweight="bold")
    axs[0].set_xlabel("Time (s)")
    axs[0].set_ylabel("Amplitude")
    axs[0].grid(True, alpha=0.3)
    
    # 2. Mel-Spectrogram
    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=2048, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img2 = librosa.display.specshow(mel_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=axs[1], cmap='magma')
    fig.colorbar(img2, ax=axs[1], format="%+2.0f dB")
    axs[1].set_title("2. Mel-Frequency Spectrogram (dB)", fontweight="bold")
    
    # 3. MFCC
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40, n_fft=2048, hop_length=512)
    img3 = librosa.display.specshow(mfcc, sr=sr, hop_length=512, x_axis='time', ax=axs[2], cmap='coolwarm')
    fig.colorbar(img3, ax=axs[2])
    axs[2].set_title("3. MFCC (40 Coefficients)", fontweight="bold")
    axs[2].set_ylabel("MFCC Coeffs")
    
    plt.tight_layout()
    plt.show()

inspect_audio_sample()

## 3. Dataset Setup & Audio Slicing

**Why Slicing?**
The GTZAN dataset has 1,000 tracks. By slicing each 30-second track into **10 segments of 3 seconds**, we augment our training data to **10,000 samples**!

If you haven't downloaded the GTZAN dataset yet, the code below automatically creates a mock GTZAN dataset so all cells run seamlessly.

In [ ]:
DATASET_PATH = "data/genres_original"
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

# Ensure dataset exists (generate mock data if empty)
if not os.path.exists(DATASET_PATH) or len(os.listdir(DATASET_PATH)) == 0:
    print("GTZAN dataset not found locally. Generating mock audio dataset for demonstration...")
    import download_dataset
    download_dataset.generate_mock_dataset(target_dir=DATASET_PATH, tracks_per_genre=4)
else:
    print(f"Using existing dataset in '{DATASET_PATH}'")

## 4. Extract MFCC Features & Save to NumPy Archive

Each 3-second chunk is converted to an MFCC matrix of shape `(130, 40)`. We strictly normalize the shape so no frames are dropped or jagged.

In [ ]:
SAMPLE_RATE = 22050
TRACK_DURATION = 30
NUM_SEGMENTS = 10
SAMPLES_PER_SEGMENT = int(SAMPLE_RATE * (TRACK_DURATION / NUM_SEGMENTS))  # 66,150
EXPECTED_FRAMES = 130
N_MFCC = 40
N_FFT = 2048
HOP_LENGTH = 512

NPZ_PATH = "data/data_mfcc.npz"

# Check if features were already extracted
if os.path.exists(NPZ_PATH):
    print(f"Loading pre-extracted features from '{NPZ_PATH}'...")
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        X = data["X"]
        y = data["y"]
        mapping = data["mapping"].tolist()
else:
    print("Extracting features from audio files...")
    import dataset_prep
    dataset_prep.extract_features(dataset_path=DATASET_PATH, output_npz=NPZ_PATH)
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        X = data["X"]
        y = data["y"]
        mapping = data["mapping"].tolist()

print(f"Extracted {X.shape[0]} segments. Shape per sample: {X.shape[1:]} (time_frames, n_mfcc)")
print(f"Genres ({len(mapping)}): {mapping}")

## 5. Train / Validation Split (80/20) & Tensor Reshaping

We split the data into **80% training** and **20% validation** (stratified by genre), and reshape $X$ into 4D `(samples, 130, 40, 1)` for 2D Convolutions.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Add channel dimension for Conv2D
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]

print(f"Train samples      : {X_train.shape[0]} (80%)")
print(f"Validation samples : {X_val.shape[0]} (20%)")
print(f"CNN Input tensor   : {X_train.shape[1:]}")

## 6. Build 2D CNN Model (Keras Functional API)

Using the Functional API guarantees universal compatibility across Keras 2 and Keras 3 (TensorFlow 2.16+).

In [ ]:
def build_genre_cnn(input_shape, num_classes=10):
    inputs = layers.Input(shape=input_shape, name="audio_input")
    
    # Block 1
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same")(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 2
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same")(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 3
    x = layers.Conv2D(128, (2, 2), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), strides=(2, 2), padding="same")(x)
    x = layers.Dropout(0.30)(x)
    
    # Dense Classifier
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="genre_probabilities")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="MusicGenreCNN")
    return model

input_shape = (X_train.shape[1], X_train.shape[2], 1)
model = build_genre_cnn(input_shape, num_classes=len(mapping))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Model Training with Callbacks

In [ ]:
os.makedirs("models", exist_ok=True)

training_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint("models/music_genre_cnn.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=32,
    callbacks=training_callbacks,
    verbose=1
)

## 8. Evaluation: Accuracy/Loss Curves & Confusion Matrix

In [ ]:
# Plot Accuracy & Loss
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(history.history["accuracy"], label="Train Accuracy", color="#2563eb", lw=2)
axs[0].plot(history.history["val_accuracy"], label="Validation Accuracy", color="#ea580c", lw=2)
axs[0].set_title("Training vs Validation Accuracy", fontweight="bold")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Accuracy")
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot(history.history["loss"], label="Train Loss", color="#2563eb", lw=2)
axs[1].plot(history.history["val_loss"], label="Validation Loss", color="#ea580c", lw=2)
axs[1].set_title("Training vs Validation Loss", fontweight="bold")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("Loss")
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report & Confusion Matrix
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_val, y_pred, target_names=mapping))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=mapping, yticklabels=mapping)
plt.title("GTZAN Genre Classification - Confusion Matrix", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.show()

## 9. Audio Prediction Function with Confidence Score & Spectrogram

In [ ]:
def classify_audio_track(file_path, model, mapping):
    """Predicts genre from audio file with confidence score and dual visualization."""
    signal, sr = librosa.load(file_path, sr=22050)
    seg_len = int(sr * 3)
    num_segs = max(1, int(len(signal) / seg_len))
    
    if len(signal) < seg_len:
        signal_padded = np.pad(signal, (0, seg_len - len(signal)), mode="constant")
    else:
        signal_padded = signal
        
    chunks = []
    for s in range(num_segs):
        chunk = signal_padded[s * seg_len : (s + 1) * seg_len]
        if len(chunk) < 2048:
            continue
        mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=40, n_fft=2048, hop_length=512).T
        if mfcc.shape[0] < 130:
            mfcc = np.pad(mfcc, ((0, 130 - mfcc.shape[0]), (0, 0)), mode="constant")
        else:
            mfcc = mfcc[:130, :]
        chunks.append(mfcc)
        
    X_in = np.array(chunks, dtype=np.float32)[..., np.newaxis]
    preds = model.predict(X_in, verbose=0)
    avg_probs = np.mean(preds, axis=0)
    winner_idx = int(np.argmax(avg_probs))
    winner = mapping[winner_idx]
    confidence = avg_probs[winner_idx] * 100.0
    
    print("=" * 60)
    print(f"⭐ PREDICTED GENRE : {winner.upper()}")
    print(f"🎯 CONFIDENCE SCORE: {confidence:.2f}%")
    print("=" * 60)
    
    # Dual Visualization: Mel-Spectrogram & Confidence Bar Chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Spectrogram
    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=2048, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, hop_length=512, x_axis="time", y_axis="mel", ax=ax1, cmap="magma")
    ax1.set_title(f"Mel-Spectrogram ({os.path.basename(file_path)})")
    
    # Confidence Bars
    y_pos = np.arange(len(mapping))
    colors = ["#22c55e" if g == winner else "#94a3b8" for g in mapping]
    ax2.barh(y_pos, avg_probs * 100, color=colors)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(mapping)
    ax2.invert_yaxis()
    ax2.set_xlabel("Probability (%)")
    ax2.set_xlim(0, 100)
    ax2.set_title(f"Confidence Scores (Winner: {winner.upper()} - {confidence:.1f}%)")
    ax2.grid(True, axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return winner, confidence

# Test on an existing sample file from the dataset
sample_files = []
for r, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith(".wav"):
            sample_files.append(os.path.join(r, f))

if sample_files:
    classify_audio_track(sample_files[0], model, mapping)